# 📧 Group Email Summarizer — Enron Dataset
**Full NLP pipeline: Load → Parse → Group → Summarise → Dashboard**

```
Enron emails.csv
      │
      ▼
EmailLoader   ─── parse RFC-2822 → group by subject thread
      │
      ▼
NLP Engine
  ├── sumy LSA       → extractive summary
  ├── KeyBERT        → key topics
  ├── VADER          → sentiment
  ├── spaCy NER      → owner detection
  └── Regex          → action items / follow-ups
      │
      ▼
pandas DataFrame  →  Excel Dashboard  +  Streamlit App
```

---
**No external API keys required. Runs entirely locally.**

## 0 — Install Dependencies

In [1]:
!pip install sumy keybert vaderSentiment spacy streamlit plotly pandas openpyxl python-dateutil -q
!python -m spacy download en_core_web_sm -q
print('✓ All dependencies installed')

  error: subprocess-exited-with-error
  
  pip subprocess to install build dependencies did not run successfully.
  exit code: 1
  
  [117 lines of output]
  Ignoring numpy: markers 'python_version >= "3.9"' don't match your environment
    Using cached setuptools-75.3.4-py3-none-any.whl.metadata (6.9 kB)
    Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
    Using cached cymem-2.0.11.tar.gz (10 kB)
    Installing build dependencies: started
    Installing build dependencies: finished with status 'done'
    Getting requirements to build wheel: started
    Getting requirements to build wheel: finished with status 'done'
    Preparing metadata (pyproject.toml): started
    Preparing metadata (pyproject.toml): finished with status 'done'
    Using cached preshed-3.0.12.tar.gz (15 kB)
    Installing build dependencies: started
    Installing build dependencies: finished with status 'error'
    error: subprocess-exited-with-error
  
    pip subprocess to install build de

✓ All dependencies installed


C:\Users\DAKSHI\AppData\Local\Programs\Python\Python38\python.exe: No module named spacy


## 1 — Imports & Config

In [2]:
import sys, re, json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# Add project root to path so utils imports work
sys.path.insert(0, str(Path('.').resolve()))

from utils.email_loader   import load_emails, group_into_threads, get_sample_threads
from utils.nlp_engine     import analyse_thread
from utils.excel_exporter import export_to_excel

# Config
USE_SAMPLE_DATA = True        # Set False to use real Enron CSV
ENRON_CSV_PATH  = 'data/emails.csv'
NROWS           = 80_000
TOP_N_THREADS   = 30

print('✓ Imports ready')

✓ Imports ready


## 2 — Load Email Data

In [3]:
if USE_SAMPLE_DATA or not Path(ENRON_CSV_PATH).exists():
    print('Using built-in sample threads (demo mode)')
    threads = get_sample_threads()
else:
    print(f'Loading from: {ENRON_CSV_PATH}')
    email_df = load_emails(csv_path=ENRON_CSV_PATH, nrows=NROWS)
    threads  = group_into_threads(email_df, min_emails=2, top_n=TOP_N_THREADS)

print(f'\n✓ {len(threads)} threads ready for analysis')
print('\nThread keys:')
for i, key in enumerate(list(threads.keys())[:5], 1):
    print(f'  {i}. {key}')
print('  ...')

Using built-in sample threads (demo mode)

✓ 10 threads ready for analysis

Thread keys:
  1. california power crisis response
  2. q4 gas contract renewal
  3. broadband unit wind-down
  4. raptor spv disclosure
  5. ferc investigation response
  ...


## 3 — Preview a Sample Thread

In [4]:
# Inspect the first thread's emails
first_key = list(threads.keys())[0]
first_emails = threads[first_key]

print(f'Thread: "{first_key}"  ({len(first_emails)} emails)\n')
print('=' * 60)
for i, e in enumerate(first_emails, 1):
    print(f'Email {i}:')
    print(f'  From   : {e["from_"]}')
    print(f'  Subject: {e["subject"]}')
    print(f'  Body   : {e["body"][:200]}…')
    print()

Thread: "california power crisis response"  (4 emails)

Email 1:
  From   : tim.belden@enron.com
  Subject: California Power Crisis Response
  Body   : Team — urgent action needed. FERC is requesting all trading logs by Friday. Please compile every ISO-NE position we held in August. Jeff, can you lead this? We need to draft a regulatory response imme…

Email 2:
  From   : jeff.dasovich@enron.com
  Subject: Re: California Power Crisis Response
  Body   : Understood. I'll coordinate with legal. We also need to loop in the government affairs team. Follow up: schedule call with FERC liaison by Wednesday. I'll have the preliminary report ready by Thursday…

Email 3:
  From   : richard.shapiro@enron.com
  Subject: Re: California Power Crisis Response
  Body   : Agreed. Also please note that the Senate committee wants written testimony. We should prepare talking points. Need to send the draft to lobbyists by Tuesday.…

Email 4:
  From   : tim.belden@enron.com
  Subject: Re: California Power C

## 4 — Run NLP Analysis on All Threads

In [5]:
results = []
total   = len(threads)

for i, (key, emails) in enumerate(threads.items(), 1):
    print(f'[{i:02d}/{total}] {key[:55]}')
    result = analyse_thread(key, emails)
    results.append(result)

df = pd.DataFrame(results)
print(f'\n✓ Done — {len(df)} threads analysed')

[01/10] california power crisis response
[02/10] q4 gas contract renewal
[03/10] broadband unit wind-down
[04/10] raptor spv disclosure
[05/10] ferc investigation response
[06/10] enron online performance review
[07/10] employee stock plan changes
[08/10] risk committee var limit breach
[09/10] india dabhol power project
[10/10] trading floor systems upgrade

✓ Done — 10 threads analysed


## 5 — Display Results Table

In [6]:
SENT_COLORS = {
    'urgent'  : ('background-color:#fdecea; color:#c0392b', '🔴'),
    'negative': ('background-color:#fef0f0; color:#922b21', '🟠'),
    'positive': ('background-color:#eafaf1; color:#1e8449', '🟢'),
    'neutral' : ('background-color:#f8f9fa; color:#2c3e50', '⚪'),
}

def style_row(row):
    s = str(row.get('Sentiment','')).lower()
    css, _ = SENT_COLORS.get(s, ('', ''))
    return [css] * len(row)

display_cols = ['Email Thread', 'Key Topic', 'Action Items', 'Owner', 'Sentiment', 'Email Count']
df_display   = df[[c for c in display_cols if c in df.columns]].copy()

# Add emoji to sentiment
df_display['Sentiment'] = df_display['Sentiment'].apply(
    lambda s: f"{SENT_COLORS.get(s.lower(), ('','⚪'))[1]} {s.capitalize()}"
)

styled = (
    df_display.style
    .apply(style_row, axis=1)
    .set_caption('📧 Group Email Intelligence — Thread Analysis')
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#1f4e79'),('color','white'),
                  ('font-weight','bold'),('padding','10px 14px'),
                  ('font-size','11px'),('text-transform','uppercase'),
                  ('letter-spacing','.5px')]
    },{
        'selector': 'td',
        'props': [('padding','9px 14px'),('font-size','12px'),
                  ('vertical-align','top'),('max-width','300px'),
                  ('word-wrap','break-word')]
    },{
        'selector': 'caption',
        'props': [('font-size','14px'),('font-weight','bold'),
                  ('color','#1f4e79'),('margin-bottom','10px')]
    }])
    .set_properties(**{'text-align': 'left'})
)

display(styled)

,Email Thread,Key Topic,Action Items,Owner,Sentiment,Email Count
0,California Power Crisis Response,"trading logs, jeff dasovich, action",compile every ISO-NE position we held in August | note that the Senate committee wants written testimony | compile trading logs by Thursday EOD | prepare executive briefing pack | schedule call with FERC liaison by Wednesday,Tim Belden,🔴 Urgent,4
1,Q4 Gas Contract Renewal,"louise kitchen, term sheet, force majeure",review and send comments by Friday | Louise to circulate revised term sheet | Sara to provide redlined agreement by EOD | John to draft counter-proposal | review the term sheet,Louise Kitchen,🔴 Urgent,3
2,Broadband Unit Wind Down,"jeff skilling, warn notices, press release",Cindy to prepare WARN notices and severance schedule | Legal to review all 47 customer contracts | Mark to circulate draft press release for review | go out 60 days before termination date | Jeff Skilling,Jeff Skilling,⚪ Neutral,4
3,Raptor Spv Disclosure,"andrew fastow, finalise disclosure, disclosure language",finalise disclosure language | Rick Causey to present revised disclosures to audit committee Tuesday | James to brief CEO and General Counsel by Monday | finalise the 10-Q disclosure language for the Raptor vehicles | be briefed immediately,Andrew Fastow,🔴 Urgent,3
4,Ferc Investigation Response,"trading, trading records, outside counsel","Legal to coordinate document collection | Tim to provide annotated trading log by Friday | all team members to brief outside counsel by Wednesday | prepare full disclosure memo by Tuesday | compile all trading records, communications, and strategy documents",Rex Rogers,🔴 Urgent,4
5,Enron Online Performance Review,"louise kitchen, during peak, peak hours",prepare the October metrics report for the board presentation | IT team to diagnose latency issue by next Wednesday | October metrics report due November 5 | Kevin to implement temporary workaround by Thursday | Louise Kitchen,Louise Kitchen,⚪ Neutral,3
6,Employee Stock Plan Changes,"cindy olson, cindy, draft communications",ensure the messaging is positive and focuses on the company's strong fundamentals | Cindy to draft communications | legal review of communications by Friday | Ken to record video message for employees | Cindy Olson,Cindy Olson,⚪ Neutral,3
7,Risk Committee Var Limit Breach,"rick urgent, trading desk, rick",Reset VaR limits and present remediation plan | Greg to coordinate position unwind with trading desk Sunday | file 8-K disclosure by Tuesday | unwind at least 40% of the position before Monday open | position unwind with trading desk Sunday,Rick Buy,🔴 Urgent,3
8,India Dabhol Power Project,"rebecca mark, rebecca, mark",Rebecca to lead arbitration proceedings | engage Goldman Sachs to explore sale options | James to coordinate with State Dept by end of week | Rebecca Mark | with State Dept by end of week,Rebecca Mark,⚪ Neutral,3
9,Trading Floor Systems Upgrade,"scott yeager, scott, yeager",IT to complete workstation deployment by Sunday night | CTO sign-off by Friday | all traders to attend mandatory training | back up their local files by Friday | Scott Yeager,Scott Yeager,🟢 Positive,3


## 6 — Deep Dive: Full Thread Analysis

In [7]:
# Show full analysis for the first thread
sample = results[0]

html = f"""
<div style="font-family:Arial,sans-serif;max-width:800px">
  <div style="background:#1f4e79;color:white;padding:14px 20px;border-radius:8px 8px 0 0">
    <strong style="font-size:15px">📧 {sample['Email Thread']}</strong>
    <span style="float:right;font-size:11px;opacity:.7">{sample['Email Count']} emails · {sample['Latest Date']}</span>
  </div>
  <div style="border:1px solid #ddd;border-top:none;border-radius:0 0 8px 8px;padding:16px 20px">

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Key Topic</div>
      <div style="font-size:13px;font-weight:bold;color:#1a5276">{sample['Key Topic']}</div>
    </div>

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Summary</div>
      <div style="font-size:12px;color:#2c3e50;line-height:1.6">{sample['Summary']}</div>
    </div>

    <div style="margin-bottom:14px">
      <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Action Items</div>
      {''.join(f"<div style='background:#f0f4ff;border-left:3px solid #5865f2;padding:6px 10px;margin:3px 0;font-size:11px'>▸ {a}</div>" for a in sample.get('Action List',[]) or sample['Action Items'].split(' | '))}
    </div>

    <div style="display:flex;gap:24px">
      <div>
        <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Owner</div>
        <div style="font-size:13px;color:#6c3483;font-weight:bold">{sample['Owner']}</div>
      </div>
      <div>
        <div style="font-size:10px;color:#7f8c8d;text-transform:uppercase;letter-spacing:.5px">Sentiment</div>
        <div style="font-size:13px">{sample['Sentiment'].capitalize()}</div>
      </div>
    </div>
  </div>
</div>
"""
display(HTML(html))

## 7 — Summary Statistics

In [8]:
sent_counts  = df['Sentiment'].value_counts()
action_count = (df['Action Items'] != 'None identified').sum()

print('=' * 50)
print('  EMAIL THREAD SUMMARY STATISTICS')
print('=' * 50)
print(f'  Total threads analysed : {len(df)}')
print(f'  Total emails processed : {int(df["Email Count"].sum())}')
print(f'  Avg emails per thread  : {df["Email Count"].mean():.1f}')
print(f'  Threads with actions   : {int(action_count)}')
print()
print('  Sentiment Breakdown:')
for sent, cnt in sent_counts.items():
    bar = '█' * cnt
    print(f'    {sent.capitalize():<12} {cnt:>3}  {bar}')
print('=' * 50)

  EMAIL THREAD SUMMARY STATISTICS
  Total threads analysed : 10
  Total emails processed : 33
  Avg emails per thread  : 3.3
  Threads with actions   : 10

  Sentiment Breakdown:
    Urgent         5  █████
    Neutral        4  ████
    Positive       1  █


## 8 — Export to Excel Dashboard

In [9]:
output_path = 'email_dashboard.xlsx'
export_to_excel(df, output_path=output_path)

# Also save CSV
df.to_csv('data/results.csv', index=False)
print('✓ CSV saved → data/results.csv')

display(HTML(f'<a href="{output_path}" style="color:#1a5276;font-size:14px">📥 Download {output_path}</a>'))

✓ Excel dashboard saved → email_dashboard.xlsx
✓ CSV saved → data/results.csv
